In [0]:
spark.sql("SHOW CREATE TABLE bus_telemetry").show(truncate=False)

# SIMULAÇÃO DE CENÁRIOS - SISTEMA STCP


## Configuração Inicial


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# REGRA 1: Limites de Potência
POTENCIA_MAXIMA_ESTACAO = 150.0  # kW
BATERIA_MAXIMA_AUTOCARRO = 350.0  # kWh
BATERIA_MINIMA_SEGURANCA = 20.0  # %
BATERIA_IDEAL = 80.0  # %

# REGRA 3: Limites de Temperatura
TEMP_BATERIA_MAX = 45.0  # °C
TEMP_BATERIA_IDEAL_MIN = 15.0  # °C
TEMP_BATERIA_IDEAL_MAX = 35.0  # °C
TEMP_CARREGADOR_MAX = 60.0  # °C

# REGRA 5: Distância ao Depósito
DEPOT_LAT = 41.183580
DEPOT_LON = -8.618978
CONSUMO_POR_KM = 1.2  # kWh/km
MARGEM_SEGURANCA = 1.2  # 20% extra

# REGRA 7: Balanceamento de Carga
POTENCIA_TOTAL_DISPONIVEL = 450.0  # kW (3 estações)
LIMITE_CARGA_REDE = 450.0  # kW

# REGRA 8: Saúde da Bateria
FAIXA_IDEAL_MIN = 20.0  # %
FAIXA_IDEAL_MAX = 80.0  # %

print("="*80)
print("CONFIGURAÇÃO CARREGADA")
print("="*80)
print(f"Potência máxima por estação: {POTENCIA_MAXIMA_ESTACAO} kW")
print(f"Limite da rede elétrica: {LIMITE_CARGA_REDE} kW")
print(f"Depósito: ({DEPOT_LAT}, {DEPOT_LON})")
print("="*80)

In [ ]:
# Criar tabela de prioridades (necessária para todos os cenários)
spark.sql("DROP TABLE IF EXISTS route_priorities")

prioridades = spark.createDataFrame([
    ("BUS-001", "200", "Bolhao - Castelo Queijo", "Alta", 9.85, 30.0),
    ("BUS-002", "201", "Aliados - Viso", "Normal", 9.78, 30.0),
    ("BUS-003", "202", "Aliados - Passeio Alegre", "Normal",  5.31, 20.0)
], ["device_id", "route_id", "route_name", "priority", "distance_km", "min_battery_required"])

prioridades.write.mode("overwrite").saveAsTable("route_priorities")

print("Prioridades configuradas!")

---

## CENÁRIO BASE (Referência)

Situação atual com dados reais sem modificações.

**Autocarros:**
- BUS-001: 30% bateria, Rota Alta
- BUS-002: 70% bateria, Rota Normal
- BUS-003: 25% bateria, Rota Normal

**Estações:**
- CS-001: IDLE, disponível
- CS-002: IDLE, disponível
- CS-003: IDLE, disponível

**Resultado esperado:**
- BUS-003 e BUS-001 devem carregar
- BUS-002 não necessita

In [ ]:
print("="*80)
print("CENÁRIO BASE - DADOS REAIS")
print("="*80)

# Usar dados reais diretamente
print("\n--- ESTADO DOS AUTOCARROS ---")
display(spark.sql("""
    SELECT 
        device_id,
        ROUND(batteryLevel, 1) as bateria_percent,
        ROUND(batteryKwh, 1) as bateria_kwh,
        ROUND(batteryTemperature, 1) as temperatura_C,
        state as estado
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_raw
    ) WHERE rn = 1
    ORDER BY device_id
"""))

print("\n--- ESTADO DAS ESTAÇÕES ---")
display(spark.sql("""
    SELECT 
        device_id,
        state as estado,
        ROUND(currentPower, 1) as potencia_kw,
        ROUND(chargerTemperature, 1) as temperatura_C,
        connectedBus
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    ) WHERE rn = 1
    ORDER BY device_id
"""))

In [ ]:
# Calcular decisões no cenário base
print("\n--- DECISÕES NO CENÁRIO BASE ---")

# Calcular scores e prioridades
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW autocarros_base AS
    WITH latest_bus AS (
        SELECT 
            b.device_id,
            ROUND(b.batteryLevel, 1) as bateria_percent,
            ROUND(b.batteryKwh, 1) as bateria_kwh,
            ROUND(b.batteryTemperature, 1) as temperatura_C,
            b.latitude,
            b.longitude,
            r.route_name,
            r.priority,
            r.min_battery_required,
            ROW_NUMBER() OVER (PARTITION BY b.device_id ORDER BY b.event_time DESC) as rn
        FROM bus_telemetry_raw b
        JOIN route_priorities r ON b.device_id = r.device_id
    ),
    with_calcs AS (
        SELECT 
            *,
            ROUND(
                111.0 * SQRT(
                    POW(({DEPOT_LAT} - latitude), 2) + 
                    POW(({DEPOT_LON} - longitude) * COS(latitude * 3.14159 / 180), 2)
                ), 2
            ) as distancia_depot_km,
            ROUND(bateria_kwh / {CONSUMO_POR_KM}, 1) as autonomia_km
        FROM latest_bus WHERE rn = 1
    ),
    with_requirements AS (
        SELECT 
            *,
            ROUND(
                ((distancia_depot_km * {CONSUMO_POR_KM} * {MARGEM_SEGURANCA} / {BATERIA_MAXIMA_AUTOCARRO}) * 100) + min_battery_required,
                1
            ) as bateria_total_necessaria
        FROM with_calcs
    ),
    with_scores AS (
        SELECT 
            *,
            CASE WHEN priority = 'Critica' THEN 40 WHEN priority = 'Alta' THEN 30 ELSE 20 END +
            CASE WHEN bateria_percent < bateria_total_necessaria THEN 60 
                 WHEN bateria_percent < (bateria_total_necessaria + 10) THEN 30 ELSE 0 END +
            CASE WHEN autonomia_km < (distancia_depot_km * {MARGEM_SEGURANCA}) THEN 100
                 WHEN autonomia_km < (distancia_depot_km * 1.5) THEN 60 ELSE 0 END
            as score_total,
            CASE 
                WHEN temperatura_C > {TEMP_BATERIA_MAX} THEN 'BLOQUEADO'
                WHEN bateria_percent > {FAIXA_IDEAL_MAX} THEN 'BLOQUEADO'
                WHEN bateria_percent >= (bateria_total_necessaria + 5) THEN 'NAO NECESSARIO'
                ELSE 'APTO'
            END as status
        FROM with_requirements
    )
    SELECT * FROM with_scores
""")

display(spark.sql("""
    SELECT 
        device_id,
        route_name,
        priority as prioridade,
        bateria_percent,
        bateria_total_necessaria,
        score_total,
        status
    FROM autocarros_base
    ORDER BY score_total DESC
"""))

# Disponibilidade estações
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW estacoes_base AS
    SELECT 
        device_id,
        state,
        CASE WHEN state = 'available' AND chargerTemperature < {TEMP_CARREGADOR_MAX} THEN 1 ELSE 0 END as disponivel
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_raw
    ) WHERE rn = 1
""")

# Alocações
print("\n--- ALOCAÇÕES BASE ---")
display(spark.sql("""
    WITH aptos AS (
        SELECT device_id, route_name, bateria_percent, score_total,
               ROW_NUMBER() OVER (ORDER BY score_total DESC) as ordem
        FROM autocarros_base WHERE status = 'APTO'
    ),
    disponiveis AS (
        SELECT device_id, ROW_NUMBER() OVER (ORDER BY device_id) as ordem
        FROM estacoes_base WHERE disponivel = 1
    )
    SELECT 
        a.device_id as autocarro,
        a.route_name,
        a.bateria_percent,
        a.score_total,
        d.device_id as estacao_alocada,
        CASE WHEN d.device_id IS NOT NULL THEN 'ALOCADO' ELSE 'FILA ESPERA' END as status
    FROM aptos a
    LEFT JOIN disponiveis d ON a.ordem = d.ordem
    ORDER BY a.score_total DESC
"""))

print("\nCenário base executado! Use como referência para comparar.")

## CENÁRIO 1: Estação CS-001 Ocupada


In [ ]:
print("="*80)
print("CENÁRIO 1: CS-001 OCUPADA")
print("="*80)

# Criar views simuladas
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bus_telemetry_c1 AS
    SELECT * FROM bus_telemetry_raw
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW charger_telemetry_c1 AS
    SELECT 
        device_id,
        CASE WHEN device_id = 'CS-001' THEN 'CHARGING' ELSE state END as state,
        CASE WHEN device_id = 'CS-001' THEN 120.0 ELSE currentPower END as currentPower,
        CASE WHEN device_id = 'CS-001' THEN 'BUS-004' ELSE connectedBus END as connectedBus,
        chargerTemperature,
        currentEfficiency,
        energyDelivered,
        totalEnergyDelivered,
        latitude,
        longitude,
        event_time,
        enqueued_time
    FROM charger_telemetry_raw
""")

print("\n--- ESTAÇÕES NO CENÁRIO 1 ---")
display(spark.sql("""
    SELECT 
        device_id,
        state,
        ROUND(currentPower, 1) as potencia_kw,
        connectedBus,
        CASE 
            WHEN state = 'available' THEN 'DISPONIVEL'
            WHEN state = 'CHARGING' THEN 'OCUPADO'
            ELSE 'INDISPONIVEL'
        END as status
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_c1
    ) WHERE rn = 1
    ORDER BY device_id
"""))

# Disponibilidade
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW estacoes_c1 AS
    SELECT 
        device_id,
        state,
        CASE WHEN state = 'available' AND chargerTemperature < {TEMP_CARREGADOR_MAX} THEN 1 ELSE 0 END as disponivel
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_c1
    ) WHERE rn = 1
""")

print("\n--- RESUMO DISPONIBILIDADE ---")
display(spark.sql("""
    SELECT 
        COUNT(*) as total_estacoes,
        SUM(disponivel) as estacoes_livres,
        SUM(CASE WHEN state = 'CHARGING' THEN 1 ELSE 0 END) as estacoes_ocupadas
    FROM estacoes_c1
"""))

In [ ]:
# Alocações no cenário 1 COM BALANCEAMENTO
print("\n--- VERIFICAR BALANCEAMENTO DE CARGA ---")

# Calcular carga atual (CS-001 já ocupada)
carga_atual = spark.sql("""
    SELECT COALESCE(SUM(currentPower), 0.0) as total
    FROM (
        SELECT currentPower,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_c1
    ) WHERE rn = 1
""").collect()[0]['total']

print(f"Carga atual na rede: {carga_atual} kW")
print(f"Limite da rede: {LIMITE_CARGA_REDE} kW")

carga_disponivel = LIMITE_CARGA_REDE - carga_atual
print(f"Carga disponível: {carga_disponivel} kW")

print("\n--- ALOCAÇÕES COM BALANCEAMENTO ---")

display(spark.sql(f"""
    WITH aptos AS (
        SELECT 
            device_id, 
            route_name, 
            bateria_percent, 
            score_total,
            ROW_NUMBER() OVER (ORDER BY score_total DESC) as ordem
        FROM autocarros_base 
        WHERE status = 'APTO'
    ),
    disponiveis AS (
        SELECT 
            device_id, 
            ROW_NUMBER() OVER (ORDER BY device_id) as ordem
        FROM estacoes_c1 
        WHERE disponivel = 1
    ),
    alocacoes AS (
        SELECT 
            a.device_id as autocarro,
            a.route_name,
            a.bateria_percent,
            a.score_total,
            a.ordem,
            d.device_id as estacao_alocada
        FROM aptos a
        LEFT JOIN disponiveis d ON a.ordem = d.ordem
    ),
    com_potencia AS (
        SELECT 
            *,
            CASE 
                WHEN estacao_alocada IS NULL THEN 0.0
                WHEN ordem = 1 THEN 
                    LEAST(150.0, {carga_disponivel})
                WHEN ordem = 2 THEN
                    CASE 
                        WHEN {carga_disponivel} >= 300 THEN 150.0
                        WHEN {carga_disponivel} > 150 THEN {carga_disponivel} - 150.0
                        ELSE 0.0
                    END
                ELSE 0.0
            END as potencia_alocada_kw,
            CASE 
                WHEN estacao_alocada IS NULL THEN 'FILA ESPERA'
                WHEN ordem = 1 THEN 'ALOCADO'
                WHEN ordem = 2 AND {carga_disponivel} > 150 THEN 'ALOCADO (Potencia reduzida)'
                WHEN ordem = 2 AND {carga_disponivel} <= 150 THEN 'FILA ESPERA (Rede saturada)'
                ELSE 'FILA ESPERA'
            END as status_final
        FROM alocacoes
    )
    SELECT 
        autocarro,
        route_name,
        bateria_percent,
        score_total,
        estacao_alocada,
        ROUND(potencia_alocada_kw, 1) as potencia_kw,
        status_final as status
    FROM com_potencia
    ORDER BY score_total DESC
"""))

print("\n--- VERIFICAÇÃO FINAL DA REDE ---")

total_alocado = spark.sql(f"""
    WITH aptos AS (
        SELECT device_id, score_total,
               ROW_NUMBER() OVER (ORDER BY score_total DESC) as ordem
        FROM autocarros_base WHERE status = 'APTO'
    ),
    disponiveis AS (
        SELECT device_id, ROW_NUMBER() OVER (ORDER BY device_id) as ordem
        FROM estacoes_c1 WHERE disponivel = 1
    ),
    alocacoes AS (
        SELECT a.ordem, d.device_id as estacao
        FROM aptos a
        LEFT JOIN disponiveis d ON a.ordem = d.ordem
    )
    SELECT 
        SUM(
            CASE 
                WHEN ordem = 1 AND estacao IS NOT NULL THEN LEAST(150.0, {carga_disponivel})
                WHEN ordem = 2 AND estacao IS NOT NULL AND {carga_disponivel} > 150 THEN {carga_disponivel} - 150.0
                ELSE 0.0
            END
        ) as potencia_total_alocada
    FROM alocacoes
""").collect()[0]['potencia_total_alocada']

carga_total_final = carga_atual + (float(total_alocado) if total_alocado else 0)

display(spark.sql(f"""
    SELECT 
        {carga_atual} as carga_pre_existente_kw,
        {total_alocado if total_alocado else 0} as carga_nova_alocada_kw,
        {carga_total_final} as carga_total_rede_kw,
        {LIMITE_CARGA_REDE} as limite_rede_kw,
        CASE 
            WHEN {carga_total_final} > {LIMITE_CARGA_REDE} THEN 'SOBRECARGA - ERRO'
            WHEN {carga_total_final} > {LIMITE_CARGA_REDE * 0.9} THEN 'ALERTA - Proximo do limite'
            ELSE 'OK - Dentro do limite'
        END as status_rede
"""))

print("\n--- COMPARAÇÃO COM BASE ---")
print("BASE: 3 estações livres → 2 autocarros a 150 kW cada = 300 kW (limite rede: 450 kW)")
print(f"CENÁRIO 1: 2 estações livres + 120 kW já em uso")
print(f"  - Carga disponível: {carga_disponivel} kW")

if carga_disponivel >= 300:
    print(f"  - Pode alocar 2 autocarros a 150 kW cada")
elif carga_disponivel >= 150:
    print(f"  - Aloca BUS-003: 150 kW")
    print(f"  - Aloca BUS-001: {carga_disponivel - 150} kW (reduzido por balanceamento)")
else:
    print(f"  - Aloca apenas BUS-003: {carga_disponivel} kW")
    print(f"  - BUS-001: FILA DE ESPERA (rede saturada)")

print(f"\nCarga total final: {carga_total_final} kW ≤ {LIMITE_CARGA_REDE} kW ✓")

## CENÁRIO 2: BUS-001 com Bateria Crítica (15%)


In [ ]:
print("="*80)
print("CENÁRIO 2: BUS-001 BATERIA CRÍTICA (15%)")
print("="*80)

# Modificar bateria BUS-001
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bus_telemetry_c2 AS
    SELECT 
        device_id,
        CASE WHEN device_id = 'BUS-001' THEN 15.0 ELSE batteryLevel END as batteryLevel,
        CASE WHEN device_id = 'BUS-001' THEN 52.5 ELSE batteryKwh END as batteryKwh,
        batteryTemperature,
        latitude,
        longitude,
        state,
        event_time,
        enqueued_time
    FROM bus_telemetry_raw
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW charger_telemetry_c2 AS
    SELECT * FROM charger_telemetry_raw
""")

print("\n--- AUTOCARROS NO CENÁRIO 2 ---")
display(spark.sql("""
    SELECT 
        device_id,
        ROUND(batteryLevel, 1) as bateria_percent,
        ROUND(batteryKwh, 1) as bateria_kwh,
        ROUND(batteryKwh / 1.2, 1) as autonomia_km
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_c2
    ) WHERE rn = 1
    ORDER BY batteryLevel
"""))

In [0]:
# Aplicar algoritmo ao cenário 2
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW autocarros_c2 AS
    WITH latest_bus AS (
        SELECT 
            b.device_id,
            ROUND(b.batteryLevel, 1) as bateria_percent,
            ROUND(b.batteryKwh, 1) as bateria_kwh,
            ROUND(b.batteryTemperature, 1) as temperatura_C,
            b.latitude,
            b.longitude,
            r.route_name,
            r.priority,
            r.min_battery_required,
            ROW_NUMBER() OVER (PARTITION BY b.device_id ORDER BY b.event_time DESC) as rn
        FROM bus_telemetry_c2 b
        JOIN route_priorities r ON b.device_id = r.device_id
    ),
    with_calcs AS (
        SELECT 
            *,
            ROUND(
                111.0 * SQRT(
                    POW(({DEPOT_LAT} - latitude), 2) + 
                    POW(({DEPOT_LON} - longitude) * COS(latitude * 3.14159 / 180), 2)
                ), 2
            ) as distancia_depot_km,
            ROUND(bateria_kwh / {CONSUMO_POR_KM}, 1) as autonomia_km
        FROM latest_bus WHERE rn = 1
    ),
    with_requirements AS (
        SELECT 
            *,
            ROUND(
                ((distancia_depot_km * {CONSUMO_POR_KM} * {MARGEM_SEGURANCA} / {BATERIA_MAXIMA_AUTOCARRO}) * 100) + min_battery_required,
                1
            ) as bateria_total_necessaria
        FROM with_calcs
    ),
    with_scores AS (
        SELECT 
            *,
            CASE WHEN priority = 'Critica' THEN 40 WHEN priority = 'Alta' THEN 30 ELSE 20 END +
            CASE WHEN bateria_percent < bateria_total_necessaria THEN 60 
                 WHEN bateria_percent < (bateria_total_necessaria + 10) THEN 30 ELSE 0 END +
            CASE WHEN autonomia_km < (distancia_depot_km * {MARGEM_SEGURANCA}) THEN 100
                 WHEN autonomia_km < (distancia_depot_km * 1.5) THEN 60 ELSE 0 END
            as score_total,
            CASE 
                WHEN temperatura_C > {TEMP_BATERIA_MAX} THEN 'BLOQUEADO'
                WHEN bateria_percent > {FAIXA_IDEAL_MAX} THEN 'BLOQUEADO'
                WHEN bateria_percent >= (bateria_total_necessaria + 5) THEN 'NAO NECESSARIO'
                ELSE 'APTO'
            END as status
        FROM with_requirements
    )
    SELECT * FROM with_scores
""")
print("autocarros_c2 criada com sucesso")

In [0]:
# === ORDEM DE CARREGAMENTO (CLARA E VISUAL) ===
print("\n" + "="*80)
print("ORDEM DE CARREGAMENTO - QUEM CARREGA PRIMEIRO?")
print("="*80)

print("\n--- CENÁRIO BASE ---")
display(spark.sql("""
    SELECT 
        ROW_NUMBER() OVER (ORDER BY score_total DESC) as PRIORIDADE,
        device_id as AUTOCARRO,
        route_name as ROTA,
        priority as TIPO_ROTA,
        bateria_percent as BATERIA,
        score_total as SCORE,
        status as STATUS,
        CASE 
            WHEN ROW_NUMBER() OVER (ORDER BY score_total DESC) = 1 THEN 'CARREGA 1º'
            WHEN ROW_NUMBER() OVER (ORDER BY score_total DESC) = 2 THEN 'CARREGA 2º'
            WHEN status = 'APTO' THEN 'FILA ESPERA'
            ELSE 'NAO CARREGA'
        END as DECISAO
    FROM autocarros_base
    ORDER BY score_total DESC
"""))

print("\n--- CENÁRIO 2 ---")
display(spark.sql("""
    SELECT 
        ROW_NUMBER() OVER (ORDER BY score_total DESC) as PRIORIDADE,
        device_id as AUTOCARRO,
        route_name as ROTA,
        priority as TIPO_ROTA,
        bateria_percent as BATERIA,
        score_total as SCORE,
        status as STATUS,
        CASE 
            WHEN ROW_NUMBER() OVER (ORDER BY score_total DESC) = 1 THEN 'CARREGA 1º'
            WHEN ROW_NUMBER() OVER (ORDER BY score_total DESC) = 2 THEN 'CARREGA 2º'
            WHEN status = 'APTO' THEN 'FILA ESPERA'
            ELSE 'NAO CARREGA'
        END as DECISAO
    FROM autocarros_c2
    ORDER BY score_total DESC
"""))

primeiro_base = spark.sql("""
    SELECT device_id, score_total
    FROM autocarros_base
    WHERE status = 'APTO'
    ORDER BY score_total DESC
    LIMIT 1
""").collect()[0]

primeiro_c2 = spark.sql("""
    SELECT device_id, score_total
    FROM autocarros_c2
    WHERE status = 'APTO'
    ORDER BY score_total DESC
    LIMIT 1
""").collect()[0]

print("\n" + "="*80)
print("RESPOSTA DIRETA:")
print("="*80)
print(f"\nBASE: {primeiro_base['device_id']} carrega PRIMEIRO (score {primeiro_base['score_total']})")
print(f"CENÁRIO 2: {primeiro_c2['device_id']} carrega PRIMEIRO (score {primeiro_c2['score_total']})")

if primeiro_base['device_id'] == primeiro_c2['device_id']:
    print(f"\nCONCLUSÃO: A prioridade MANTEVE-SE → {primeiro_c2['device_id']} continua 1º")
else:
    print(f"\nCONCLUSÃO: A prioridade MUDOU!")
    print(f"  BASE: {primeiro_base['device_id']} era 1º")
    print(f"  CENÁRIO 2: {primeiro_c2['device_id']} é agora 1º")

print("="*80)

## CENÁRIO 3: Todas as Estações Ocupadas

**Carga total da rede: 370 kW (perto do limite de 450 kW)**


In [ ]:
print("="*80)
print("CENÁRIO 3: TODAS ESTAÇÕES OCUPADAS")
print("="*80)

spark.sql("""
    CREATE OR REPLACE TEMP VIEW bus_telemetry_c3 AS
    SELECT * FROM bus_telemetry_raw
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW charger_telemetry_c3 AS
    SELECT 
        device_id,
        'CHARGING' as state,
        CASE device_id
            WHEN 'CS-001' THEN 120.0
            WHEN 'CS-002' THEN 150.0
            WHEN 'CS-003' THEN 100.0
        END as currentPower,
        CASE device_id
            WHEN 'CS-001' THEN 'BUS-004'
            WHEN 'CS-002' THEN 'BUS-005'
            WHEN 'CS-003' THEN 'BUS-006'
        END as connectedBus,
        chargerTemperature,
        currentEfficiency,
        energyDelivered,
        totalEnergyDelivered,
        latitude,
        longitude,
        event_time,
        enqueued_time
    FROM charger_telemetry_raw
""")

print("\n--- ESTAÇÕES NO CENÁRIO 3 ---")
display(spark.sql("""
    SELECT 
        device_id,
        state,
        ROUND(currentPower, 1) as potencia_kw,
        connectedBus
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_c3
    ) WHERE rn = 1
    ORDER BY device_id
"""))

print("\n--- CARGA TOTAL DA REDE ---")
display(spark.sql("""
    SELECT 
        SUM(currentPower) as carga_total_kw,
        450.0 as limite_kw,
        ROUND((SUM(currentPower) / 450.0) * 100, 1) as percentagem_uso
    FROM (
        SELECT currentPower,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_c3
    ) WHERE rn = 1
"""))

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW estacoes_c3 AS
    SELECT 
        device_id,
        state,
        0 as disponivel
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_c3
    ) WHERE rn = 1
""")

print("\n--- FILA DE ESPERA (por ordem de prioridade) ---")
display(spark.sql("""
    SELECT 
        ROW_NUMBER() OVER (ORDER BY score_total DESC) as posicao_fila,
        device_id as autocarro,
        route_name,
        priority as prioridade,
        bateria_percent,
        score_total,
        'FILA DE ESPERA' as status
    FROM autocarros_base
    WHERE status = 'APTO'
    ORDER BY score_total DESC
"""))

print("\nIMPACTO: Sistema organiza fila de espera por prioridade.")
print("Quando estação ficar livre, BUS-003 (score mais alto) será atendido primeiro.")

## CENÁRIO 4: Temperatura Elevada em BUS-002


In [ ]:
print("="*80)
print("CENÁRIO 4: TEMPERATURA ELEVADA")
print("="*80)

spark.sql("""
    CREATE OR REPLACE TEMP VIEW bus_telemetry_c4 AS
    SELECT 
        device_id,
        batteryLevel,
        batteryKwh,
        CASE WHEN device_id = 'BUS-002' THEN 40.0 ELSE batteryTemperature END as batteryTemperature,
        latitude,
        longitude,
        state,
        event_time,
        enqueued_time
    FROM bus_telemetry_raw
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW charger_telemetry_c4 AS
    SELECT * FROM charger_telemetry_raw
""")

print("\n--- TEMPERATURAS NO CENÁRIO 4 ---")
display(spark.sql("""
    SELECT 
        device_id,
        ROUND(batteryLevel, 1) as bateria_percent,
        ROUND(batteryTemperature, 1) as temperatura_C,
        CASE 
            WHEN batteryTemperature > 45.0 THEN 'CRITICO - Parar carregamento'
            WHEN batteryTemperature > 35.0 THEN 'ALTO - Reduzir potencia'
            WHEN batteryTemperature < 15.0 THEN 'BAIXO - Pre-aquecer'
            ELSE 'IDEAL'
        END as status_temp,
        CASE 
            WHEN batteryTemperature > 45.0 THEN 0.0
            WHEN batteryTemperature > 35.0 THEN 80.0
            WHEN batteryTemperature < 15.0 THEN 50.0
            ELSE 150.0
        END as potencia_ajustada_kw
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_c4
    ) WHERE rn = 1
    ORDER BY batteryTemperature DESC
"""))

print("\nIMPACTO: BUS-002 com 40°C → Potência reduzida para 80 kW (proteção bateria)")
print("Tempo de carregamento aumenta ~87% (150kW → 80kW)")

## CENÁRIO 5: Múltiplos Problemas Simultâneos


In [ ]:
print("="*80)
print("CENÁRIO 5: MÚLTIPLOS PROBLEMAS (Pior Caso)")
print("="*80)

# Autocarros com problemas
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bus_telemetry_c5 AS
    SELECT 
        device_id,
        CASE WHEN device_id = 'BUS-001' THEN 12.0 ELSE batteryLevel END as batteryLevel,
        CASE WHEN device_id = 'BUS-001' THEN 42.0 ELSE batteryKwh END as batteryKwh,
        CASE WHEN device_id = 'BUS-003' THEN 38.0 ELSE batteryTemperature END as batteryTemperature,
        latitude,
        longitude,
        state,
        event_time,
        enqueued_time
    FROM bus_telemetry_raw
""")

# Estações com problemas
spark.sql("""
    CREATE OR REPLACE TEMP VIEW charger_telemetry_c5 AS
    SELECT 
        device_id,
        CASE device_id
            WHEN 'CS-001' THEN 'CHARGING'
            WHEN 'CS-002' THEN 'FAULT'
            ELSE state
        END as state,
        CASE device_id
            WHEN 'CS-001' THEN 150.0
            ELSE currentPower
        END as currentPower,
        CASE WHEN device_id = 'CS-001' THEN 'BUS-004' ELSE connectedBus END as connectedBus,
        chargerTemperature,
        currentEfficiency,
        energyDelivered,
        totalEnergyDelivered,
        latitude,
        longitude,
        event_time,
        enqueued_time
    FROM charger_telemetry_raw
""")

print("\n--- PROBLEMAS IDENTIFICADOS ---")
print("\nAUTOCARROS:")
display(spark.sql("""
    SELECT 
        device_id,
        ROUND(batteryLevel, 1) as bateria_percent,
        ROUND(batteryTemperature, 1) as temperatura_C,
        CASE 
            WHEN batteryLevel < 15 THEN 'EMERGENCIA - Bateria critica'
            WHEN batteryTemperature > 35 THEN 'ALERTA - Temperatura alta'
            ELSE 'OK'
        END as problema
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM bus_telemetry_c5
    ) WHERE rn = 1
    ORDER BY batteryLevel
"""))

print("\nESTAÇÕES:")
display(spark.sql("""
    SELECT 
        device_id,
        state,
        ROUND(currentPower, 1) as potencia_kw,
        connectedBus,
        CASE 
            WHEN state = 'available' THEN 'DISPONIVEL'
            WHEN state = 'CHARGING' THEN 'OCUPADO'
            WHEN state = 'FAULT' THEN 'EM MANUTENCAO'
            ELSE 'INDISPONIVEL'
        END as status
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_c5
    ) WHERE rn = 1
    ORDER BY device_id
"""))

In [ ]:
# === CENÁRIO 5: ALGORITMO + ALOCAÇÕES ===

# 1. Criar view autocarros_c5 (scoring completo com temperatura)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW autocarros_c5 AS
    WITH latest_bus AS (
        SELECT
            b.device_id,
            ROUND(b.batteryLevel, 1) as bateria_percent,
            ROUND(b.batteryKwh, 1)   as bateria_kwh,
            ROUND(b.batteryTemperature, 1) as temperatura_C,
            b.latitude, b.longitude,
            r.route_name, r.priority, r.min_battery_required,
            ROW_NUMBER() OVER (PARTITION BY b.device_id ORDER BY b.event_time DESC) as rn
        FROM bus_telemetry_c5 b
        JOIN route_priorities r ON b.device_id = r.device_id
    ),
    with_calcs AS (
        SELECT *,
            ROUND(111.0 * SQRT(
                POW(({DEPOT_LAT} - latitude), 2) +
                POW(({DEPOT_LON} - longitude) * COS(latitude * 3.14159 / 180), 2)
            ), 2) as distancia_depot_km,
            ROUND(bateria_kwh / {CONSUMO_POR_KM}, 1) as autonomia_km
        FROM latest_bus WHERE rn = 1
    ),
    with_req AS (
        SELECT *,
            ROUND(((distancia_depot_km * {CONSUMO_POR_KM} * {MARGEM_SEGURANCA} / {BATERIA_MAXIMA_AUTOCARRO}) * 100) + min_battery_required, 1) as bateria_total_necessaria
        FROM with_calcs
    ),
    with_scores AS (
        SELECT *,
            CASE WHEN priority = 'Critica' THEN 40 WHEN priority = 'Alta' THEN 30 ELSE 20 END +
            CASE WHEN bateria_percent < bateria_total_necessaria THEN 60
                 WHEN bateria_percent < (bateria_total_necessaria + 10) THEN 30 ELSE 0 END +
            CASE WHEN autonomia_km < (distancia_depot_km * {MARGEM_SEGURANCA}) THEN 100
                 WHEN autonomia_km < (distancia_depot_km * 1.5) THEN 60 ELSE 0 END
            as score_total,
            CASE
                WHEN temperatura_C > {TEMP_BATERIA_MAX} THEN 'BLOQUEADO'
                WHEN bateria_percent > {FAIXA_IDEAL_MAX} THEN 'BLOQUEADO'
                WHEN bateria_percent >= (bateria_total_necessaria + 5) THEN 'NAO NECESSARIO'
                ELSE 'APTO'
            END as status,
            -- Potência ajustada pela temperatura da bateria
            CASE
                WHEN temperatura_C > {TEMP_BATERIA_MAX} THEN 0.0
                WHEN temperatura_C > {TEMP_BATERIA_IDEAL_MAX} THEN 80.0
                WHEN temperatura_C < {TEMP_BATERIA_IDEAL_MIN} THEN 50.0
                ELSE {POTENCIA_MAXIMA_ESTACAO}
            END as potencia_temp_kw
        FROM with_req
    )
    SELECT * FROM with_scores
""")

# 2. Criar view estacoes_c5
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW estacoes_c5 AS
    SELECT
        device_id, state,
        CASE WHEN state = 'available' AND chargerTemperature < {TEMP_CARREGADOR_MAX} THEN 1 ELSE 0 END as disponivel
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_c5
    ) WHERE rn = 1
""")

# 3. Calcular carga actual e disponível
carga_atual_c5 = spark.sql("""
    SELECT COALESCE(SUM(currentPower), 0.0) as total
    FROM (
        SELECT currentPower,
               ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY event_time DESC) as rn
        FROM charger_telemetry_c5
    ) WHERE rn = 1
""").collect()[0]['total']

carga_disponivel_c5 = LIMITE_CARGA_REDE - float(carga_atual_c5)

print(f"Carga actual na rede: {float(carga_atual_c5)} kW")
print(f"Limite da rede: {LIMITE_CARGA_REDE} kW")
print(f"Carga disponível: {carga_disponivel_c5} kW")

# 4. Mostrar alocações
print("\n--- ALOCAÇÕES CENÁRIO 5 ---")
display(spark.sql(f"""
    WITH aptos AS (
        SELECT device_id, route_name, bateria_percent, temperatura_C,
               score_total, potencia_temp_kw, status,
               ROW_NUMBER() OVER (ORDER BY score_total DESC) as ordem
        FROM autocarros_c5
        WHERE status = 'APTO'
    ),
    disponiveis AS (
        SELECT device_id,
               ROW_NUMBER() OVER (ORDER BY device_id) as ordem
        FROM estacoes_c5 WHERE disponivel = 1
    )
    SELECT
        a.device_id as autocarro,
        a.route_name,
        a.bateria_percent,
        a.temperatura_C,
        a.score_total,
        ROUND(LEAST(a.potencia_temp_kw, {carga_disponivel_c5}), 1) as potencia_alocada_kw,
        d.device_id as estacao_alocada,
        CASE WHEN d.device_id IS NOT NULL THEN 'ALOCADO' ELSE 'FILA ESPERA' END as status
    FROM aptos a
    LEFT JOIN disponiveis d ON a.ordem = d.ordem
    ORDER BY a.score_total DESC
"""))


In [0]:
# Verificação final - COM CONVERSÃO DE TIPOS
print("\n--- VERIFICAÇÃO FINAL ---")

total_alocado_c5 = spark.sql(f"""
    WITH aptos AS (
        SELECT device_id, potencia_temp_kw, score_total,
               ROW_NUMBER() OVER (ORDER BY score_total DESC) as ordem
        FROM autocarros_c5 WHERE status = 'APTO'
    ),
    disponiveis AS (
        SELECT device_id, ROW_NUMBER() OVER (ORDER BY device_id) as ordem
        FROM estacoes_c5 WHERE disponivel = 1
    ),
    alocacoes AS (
        SELECT a.ordem, a.potencia_temp_kw, d.device_id as estacao
        FROM aptos a LEFT JOIN disponiveis d ON a.ordem = d.ordem
    )
    SELECT 
        CAST(COALESCE(SUM(
            CASE 
                WHEN ordem = 1 AND estacao IS NOT NULL THEN 
                    CASE WHEN potencia_temp_kw < {carga_disponivel_c5} THEN potencia_temp_kw ELSE {carga_disponivel_c5} END
                WHEN ordem = 2 AND estacao IS NOT NULL AND {carga_disponivel_c5} > potencia_temp_kw THEN
                    CASE 
                        WHEN potencia_temp_kw < ({carga_disponivel_c5} - (CASE WHEN potencia_temp_kw < {carga_disponivel_c5} THEN potencia_temp_kw ELSE {carga_disponivel_c5} END))
                        THEN potencia_temp_kw 
                        ELSE ({carga_disponivel_c5} - (CASE WHEN potencia_temp_kw < {carga_disponivel_c5} THEN potencia_temp_kw ELSE {carga_disponivel_c5} END))
                    END
                ELSE 0.0
            END
        ), 0) AS DOUBLE) as total
    FROM alocacoes
""").collect()[0]['total']

carga_total_final_c5 = float(carga_atual_c5) + float(total_alocado_c5)

display(spark.sql(f"""
    SELECT 
        {float(carga_atual_c5)} as carga_pre_existente_kw,
        {float(total_alocado_c5)} as carga_nova_kw,
        {carga_total_final_c5} as carga_total_final_kw,
        {LIMITE_CARGA_REDE} as limite_kw,
        CASE 
            WHEN {carga_total_final_c5} > {LIMITE_CARGA_REDE} THEN 'ERRO - SOBRECARGA'
            WHEN {carga_total_final_c5} > {LIMITE_CARGA_REDE * 0.9} THEN 'ALERTA - Proximo do limite'
            ELSE 'OK'
        END as status_rede
"""))

print("\n--- ANÁLISE DO CENÁRIO 5 ---")
print(f"Recursos: 1 estação livre (CS-003), {carga_disponivel_c5} kW disponíveis")
print("Autocarros: 2 precisam (BUS-001 crítico 12%, BUS-003 urgente)")
print("\nDECISÃO AUTOMÁTICA COM REGRAS:")
print("1. BUS-001 (score mais alto, 12% bateria) → CS-003")
print(f"   Potência: MIN(150, temperatura, {carga_disponivel_c5}) kW")
print("2. BUS-003 (segundo score) →")
if carga_disponivel_c5 > 150:
    print(f"   Alocado com potência reduzida: {carga_disponivel_c5 - 150} kW")
else:
    print("   FILA DE ESPERA (rede saturada)")
print(f"\nCarga total: {carga_total_final_c5} kW ≤ {LIMITE_CARGA_REDE} kW ✓")